# System 1: Property Extraction Pipeline

This notebook runs System 1 to extract material properties W required for a substitute material.

**Workflow**: Query → RAG → Questions → Answers → Extract Keywords → Properties W


## Setup: Import modules and load configuration


In [1]:
import sys
import os
import json
from pathlib import Path

# Add src to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.config import load_config
from src.utils import llm, TransformerEmbeddingFunction, custom_token_count_function, ChatLogger
from src.agents import ResearchAnalyst, ResearchManager, ResearchAssistant, ResearchScientist
from src.pipelines import run_fixed_pipeline

# ChromaDB and Graph imports
from chromadb import PersistentClient
from chromadb.config import Settings
from autogen.agentchat.contrib.vectordb.chromadb import ChromaVectorDB
import networkx as nx
from sentence_transformers import SentenceTransformer
from GraphReasoning import load_embeddings

# Load configuration
config = load_config()
print("✓ Configuration loaded")


✓ Configuration loaded


## Initialize LLM and Embeddings


In [2]:
# Initialize LLM wrapper
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"]
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
print("✓ LLM wrapper initialized")

# Initialize embedding model
embedding_tokenizer = ''  # Placeholder for SentenceTransformer
embedding_model = SentenceTransformer(config["embeddings"]["model_name"], trust_remote_code=True)
embedding_function = TransformerEmbeddingFunction(
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model
)
print("✓ Embedding model initialized")


✓ LLM wrapper initialized


<All keys matched successfully>


✓ Embedding model initialized


## Load Knowledge Graphs and Embeddings


In [3]:
# Load Material Properties Knowledge Graph (Used in Step 7)
graphs_cfg = config["data"]["graphs"]
kg_dir = graphs_cfg["kg_dir"]

mp_cfg = graphs_cfg["material_properties"]
mp_graph_path = os.path.join(kg_dir, mp_cfg["graph_file"])
G_materialproperties = nx.read_graphml(mp_graph_path)
relation = nx.get_edge_attributes(G_materialproperties, "title")
nx.set_edge_attributes(G_materialproperties, relation, "relation")
nx.set_node_attributes(G_materialproperties, nx.pagerank(G_materialproperties), "pr")
print(f"Material Properties KG loaded: {G_materialproperties}")

# Load node embeddings for Material Properties KG
mp_embeddings_path = os.path.join(kg_dir, mp_cfg["embedding_file"])
node_embeddings_materialproperties = load_embeddings(mp_embeddings_path)
print("✓ Material Properties knowledge graph and embeddings loaded")

# Load PFAS Knowledge Graph (Note: Not used in Step 7 - only Material Properties KG is used)
pfas_cfg = graphs_cfg["pfas"]
pfas_graph_path = os.path.join(kg_dir, pfas_cfg["graph_file"])
G_pfas = nx.read_graphml(pfas_graph_path)
relation_pfas = nx.get_edge_attributes(G_pfas, "title")
nx.set_edge_attributes(G_pfas, relation_pfas, "relation")
nx.set_node_attributes(G_pfas, nx.pagerank(G_pfas), "pr")
print(f"PFAS KG loaded: {G_pfas}")

# Load node embeddings for PFAS KG
pfas_embeddings_path = os.path.join(kg_dir, pfas_cfg["embedding_file"])
node_embeddings_pfas = load_embeddings(pfas_embeddings_path)
print("✓ PFAS knowledge graph and embeddings loaded (not used in Step 7)")


Material Properties KG loaded: DiGraph with 317800 nodes and 774009 edges
✓ Material Properties knowledge graph and embeddings loaded
PFAS KG loaded: DiGraph with 144335 nodes and 459248 edges
✓ PFAS knowledge graph and embeddings loaded (not used in Step 7)


## Load ChromaDB Corpus


In [4]:
# Connect to ChromaDB - PFAS Papers ChromaDB
chroma_cfg = config["data"]["chromadb"]
pfas_cfg = chroma_cfg["pfas"]
base_path = chroma_cfg.get("base_path", "")

if base_path:
    database_path = os.path.join(base_path, pfas_cfg["database_path"])
else:
    database_path = pfas_cfg["database_path"]

client = PersistentClient(path=database_path)
collection = client.get_collection(pfas_cfg["collection_name"], embedding_function=embedding_function)

ChromaDB_PFAS = ChromaVectorDB(client=client, embedding_function=embedding_function)
ChromaDB_PFAS.active_collection = collection
print("✓ ChromaDB ready to use")


✓ ChromaDB ready to use


## Initialize Agents


In [5]:
# Create chat logger for this run
import uuid
import glob
import os
from datetime import datetime

# Helper function to generate human-readable run_id
def generate_run_id(output_dir, prefix="system1"):
    """Generate human-readable run_id: YYYYMMDDHH_0 format"""
    now = datetime.now()
    base_id = now.strftime("%Y%m%d%H")  # YYYYMMDDHH
    
    # Find existing files with same base
    pattern = os.path.join(output_dir, f"{prefix}_{base_id}_*.json")
    existing_files = glob.glob(pattern)
    
    # Extract counters and find max
    max_counter = -1
    for file in existing_files:
        filename = os.path.basename(file)
        # Extract counter from filename: prefix_YYYYMMDDHH_N.json
        parts = filename.replace(".json", "").split("_")
        if len(parts) >= 3:
            try:
                counter = int(parts[-1])
                max_counter = max(max_counter, counter)
            except ValueError:
                pass
    
    # Increment counter
    counter = max_counter + 1
    return f"{base_id}_{counter}"

# Generate human-readable run_id (will be used for both chat log and pipeline log)
output_dir = "./pipeline_logs"
os.makedirs(output_dir, exist_ok=True)
run_id = generate_run_id(output_dir, prefix="system1")

chat_logger = ChatLogger(
    run_id=run_id,
    pipeline="material_requirements"
)
print(f"✓ Chat logger initialized (run_id: {run_id})")

# Initialize agents
pipeline_cfg = config["pipelines"]["material_requirements"]  # Substitute Material Requirements Analysis

analyst = ResearchAnalyst(
    collection=collection,
    embedding_function=embedding_function,
    n_results=pipeline_cfg["n_results"],
    chat_logger=chat_logger
)

manager = ResearchManager(
    name="research_manager",
    system_message=None,  # Uses default
    generate_fn=generate,
    chat_logger=chat_logger
)

research_assistant = ResearchAssistant(
    name="research_assistant",
    system_message=None,  # Uses default
    generate_fn=generate,
    chat_logger=chat_logger
)

# Initialize ResearchScientist with Material Properties knowledge graph only
# Step 7 uses only the Material Properties KG for finding connections between keywords
scientist = ResearchScientist(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    chat_logger=chat_logger
)

# Initialize PFAS ResearchScientist for querying PFAS KG during question answering (Step 4)
# This enables cross-domain relationship discovery when answering subquestions
pfas_scientist = ResearchScientist(
    knowledge_graph=G_pfas,
    node_embeddings=node_embeddings_pfas,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    chat_logger=chat_logger
)

print("✓ All agents initialized")
print("✓ ResearchScientist configured with Material Properties knowledge graph only (for Step 7)")
print("✓ PFAS ResearchScientist configured for cross-domain querying during question answering (Step 4)")


✓ Chat logger initialized (run_id: 2026021912_0)
✓ All agents initialized
✓ ResearchScientist configured with Material Properties knowledge graph only (for Step 7)
✓ PFAS ResearchScientist configured for cross-domain querying during question answering (Step 4)


## Run System 1: Extract Material Properties


In [6]:
# Define your research query
sentence = "Find a substitute material for PTFE that can be used in industrial seals applications"
material_X = "PTFE"
application_Y = "industrial seals"
keywords = [material_X, application_Y]

# Run the fixed pipeline workflow
result = run_fixed_pipeline(
    sentence=sentence,
    keywords=keywords,
    analyst=analyst,
    manager=manager,
    research_assistant=research_assistant,
    scientist=scientist,  # Material Properties KG for Step 7
    pfas_scientist=pfas_scientist,  # PFAS KG for Step 4 (question answering)
    include_rag_context=pipeline_cfg["include_rag_context"],
    max_items=pipeline_cfg["max_items"],
    temperature=config["llm"]["temperature"],
    n_results=pipeline_cfg["n_results"],
    chat_logger=chat_logger
)


Material Requirements Analysis: Starting Process

----------------------------------------------------------------------
Step 1: ResearchAnalyst - Performing RAG Retrieval
----------------------------------------------------------------------
→ Querying ChromaDB with sentence and keywords...
✓ RAG retrieval complete: 0 documents retrieved
✗ No RAG results found, using analyze_question instead (i.e., no keywords used in retrieval)
✓ RAG retrieval complete: 5 documents retrieved
  Document previews:
    1. | Auto/Aero Fuels   Polytetrafluoroethylene (PTFE)   [10,18]  Polym oly ymers   Lubricants   [10,18]   -  Viton B: Gaskets  Auto/Aero Fuels   Lubrican...
    2. ### 2.4. Applications Of Pfas In Modern Automobiles

PFAS has been widely used in high-performance mechanical systems such as vehicle combustion engin...
    3. Numerous PFAS have been used as processing aids, raw material, or manufacturing intermediate in fluoropolymer production. Fluoropolymers, which can be...

-------------

## Format Output for System 2


In [7]:
extracted_keywords = result.get("extracted_keywords", []) or []
keywords = result.get("keywords", []) or []
properties_W = {
    "required": extracted_keywords,   # List of property names
    "target_values": {}               # Populate later if values are extracted
}

print("=" * 70)
print("Material Properties W (for System 2):")
print("=" * 70)
print(f"Required properties ({len(properties_W['required'])}):")
for i, prop in enumerate(properties_W["required"], 1):
    print(f"  {i}. {prop}")

# Extract subgraph data from keyword_connections for System 2
subgraph_data = None
keyword_connections = result.get("keyword_connections")

if keyword_connections:
    # Serialize subgraph data for System 2
    subgraph_data = {
        "summary": keyword_connections.get("summary", {}) or {},
        "matched_node_ids": keyword_connections.get("matched_node_ids", []) or [],
        "found_paths": keyword_connections.get("found_paths", []) or [],
        "keyword_to_nodes": keyword_connections.get("keyword_to_nodes", {}) or {},
        "kg_results": keyword_connections.get("kg_results", {}) or {},
        "strategy": (keyword_connections.get("summary", {}) or {}).get("strategy", "single"),
    }

    # For connection_subgraph, serialize as node/edge lists (supports NetworkX graph objects)
    connection_subgraph = keyword_connections.get("connection_subgraph")
    if connection_subgraph is not None:
        try:
            # Nodes/edges
            nodes = list(connection_subgraph.nodes())
            edges = list(connection_subgraph.edges())

            # Node attributes
            node_attributes = {n: dict(connection_subgraph.nodes[n]) for n in nodes}

            # Edge attributes (MultiGraph vs Graph compatibility)
            # Convert tuple keys to strings for JSON serialization
            edge_attributes = {}
            is_multi = getattr(connection_subgraph, "is_multigraph", lambda: False)()
            if is_multi:
                # edges are (u, v, k) in MultiGraph when keys=True
                for u, v, k, data in connection_subgraph.edges(keys=True, data=True):
                    # Convert tuple key to string: "u|v|k"
                    edge_key = f"{u}|{v}|{k}"
                    edge_attributes[edge_key] = dict(data)
                edges = list(connection_subgraph.edges(keys=True))
            else:
                for u, v, data in connection_subgraph.edges(data=True):
                    # Convert tuple key to string: "u|v"
                    edge_key = f"{u}|{v}"
                    edge_attributes[edge_key] = dict(data)

            subgraph_data["connection_subgraph"] = {
                "nodes": nodes,
                "edges": edges,
                "node_attributes": node_attributes,
                "edge_attributes": edge_attributes,
            }
        except Exception as e:
            print(f"Warning: Could not serialize connection_subgraph: {e}")
            subgraph_data["connection_subgraph"] = None

    print("✓ Extracted subgraph data:")
    print(f"  - Matched nodes: {len(subgraph_data.get('matched_node_ids', []))}")
    print(f"  - Found paths: {len(subgraph_data.get('found_paths', []))}")
    if subgraph_data.get("connection_subgraph"):
        print(f"  - Subgraph nodes: {len(subgraph_data['connection_subgraph'].get('nodes', []))}")
        print(f"  - Subgraph edges: {len(subgraph_data['connection_subgraph'].get('edges', []))}")
else:
    print("⚠ No keyword_connections found in result, subgraph will not be available for System 2")

# Save for System 2
# Use the same run_id that was used for the chat logger
timestamp = datetime.utcnow().isoformat() + "Z"

output_path = os.path.join(output_dir, f"system1_{run_id}.json")

payload = {
    "run_id": run_id,
    "timestamp": timestamp,
    "sentence": result.get("sentence"),
    "keywords": result.get("keywords", []) or [],
    "material_X": material_X,
    "application_Y": application_Y,
    "properties_W": properties_W,
    "extracted_keywords": extracted_keywords,
    "num_keywords": len(extracted_keywords),
    "subgraph": subgraph_data,
    "chat_log_path": result.get("chat_log_path"),
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False, default=str)

print(f"✓ Saved properties W and subgraph to {output_path}")
print("  Use this file as input for System 2 (material discovery)")


Material Properties W (for System 2):
Required properties (32):
  1. **Key material properties, performance targets, constraints, and critical characteristics for a PTFE‑E substitute**
  2. Continuous operating temperature ≥ 250 °C
  3. Transient peak tolerance ≥ 280–290 °C
  4. Thermal degradation onset > 350 °C (mass loss < 1 % up to 350 °C)
  5. Glass‑transition far below service temperatures (≈ −80 °C for PTFE)
  6. Negligible creep at 250 °C
  7. Resistance to concentrated acids (H₂SO₄, HNO₃, HF)
  8. Hydrocarbon resistance (alkanes, aromatics)
  9. Solvent resistance (acetone, alcohols, ketones, chlorinated solvents)
  10. Oxidizer tolerance (H₂O₂, O₃, peracids)
  11. High‑temperature stability up to 300 °C
  12. UV and ionizing radiation resistance
  13. Low friction coefficient μ ≤ 0.13 under expected loads/speeds
  14. Wear rate ≤ 5 × 10⁻⁷ mm³/N·m (or comparable PTFE benchmark)
  15. Compression set < 5 % after 1,000 h at 80 °C
  16. Chemical inertness (no leaching or swelling

## Display Results Summary


In [8]:
print("\n" + "=" * 70)
print("System 1 Results Summary")
print("=" * 70)
print(f"RAG documents retrieved: {result['num_rag_results']}")
print(f"Questions generated: {result['num_manager_items']}")
print(f"Questions answered: {result['num_question_answers']}")
print(f"Keywords extracted: {result['num_extracted_keywords']}")

# Display keyword connections from both knowledge graphs
if result.get('keyword_connections'):
    conn_summary = result['keyword_connections'].get('summary', {})
    strategy = conn_summary.get('strategy', 'single')
    
    if strategy == 'separate':
        # Display results from both KGs separately
        print("\n" + "=" * 70)
        print("Keyword Connections from Knowledge Graphs")
        print("=" * 70)
        
        kg1_info = conn_summary.get('kg1', {})
        kg2_info = conn_summary.get('kg2', {})
        
        if kg1_info:
            kg1_summary = kg1_info.get('summary', {})
            print(f"\n{kg1_info.get('name', 'KG1').upper()}: {kg1_info.get('description', '')}")
            if kg1_summary.get('connections_found'):
                print(f"  ✓ Connections found: {kg1_summary.get('num_paths_found', 0)} paths")
                print(f"    Matched nodes: {kg1_summary.get('num_matched_nodes', 0)}")
                print(f"    Subgraph: {kg1_summary.get('subgraph_nodes', 0)} nodes, {kg1_summary.get('subgraph_edges', 0)} edges")
            else:
                print(f"  ✗ No connections found")
        
        if kg2_info:
            kg2_summary = kg2_info.get('summary', {})
            print(f"\n{kg2_info.get('name', 'KG2').upper()}: {kg2_info.get('description', '')}")
            if kg2_summary.get('connections_found'):
                print(f"  ✓ Connections found: {kg2_summary.get('num_paths_found', 0)} paths")
                print(f"    Matched nodes: {kg2_summary.get('num_matched_nodes', 0)}")
                print(f"    Subgraph: {kg2_summary.get('subgraph_nodes', 0)} nodes, {kg2_summary.get('subgraph_edges', 0)} edges")
            else:
                print(f"  ✗ No connections found")
        
        # Use the formatted output helper if available
        try:
            formatted_output = scientist.format_separate_results(result['keyword_connections'])
            print("\n" + formatted_output)
        except:
            pass
    else:
        # Single KG or merged strategy
        if conn_summary.get('connections_found'):
            print(f"Keyword connections: {conn_summary.get('subgraph_nodes', 0)} nodes, {conn_summary.get('subgraph_edges', 0)} edges")
            if conn_summary.get('kg1_summary') and conn_summary.get('kg2_summary'):
                print(f"  - Material Properties KG: {conn_summary['kg1_summary'].get('num_paths_found', 0)} paths")
                print(f"  - PFAS KG: {conn_summary['kg2_summary'].get('num_paths_found', 0)} paths")

print("\nGenerated Questions:")
for i, question in enumerate(result['manager_result'], 1):
    print(f"  {i}. {question}")

if result.get('extracted_keywords'):
    print(f"\nExtracted Keywords ({len(result['extracted_keywords'])}):")
    for i, kw in enumerate(result['extracted_keywords'][:10], 1):
        print(f"  {i}. {kw}")
    if len(result['extracted_keywords']) > 10:
        print(f"  ... and {len(result['extracted_keywords']) - 10} more")



System 1 Results Summary
RAG documents retrieved: 5
Questions generated: 4
Questions answered: 4
Keywords extracted: 32
Keyword connections: 241 nodes, 1824 edges

Generated Questions:
  1. What maximum operating temperatures and thermal stability limits must a substitute material meet for industrial seals that currently rely on PT F‑E, particularly in high‑temperature processes such as fuel handling or power electronics?
  2. Which specific chemical resistances (e.g., to strong acids, hydrocarbons, solvents, and oxidizing agents) are essential for PT F‑E seals used in petrochemical, oil & gas, and semiconductor manufacturing environments?
  3. What mechanical performance criteria—such as compression set, wear resistance, and low friction coefficient—must a replacement material satisfy to replicate the sealing effectiveness of PT F‑E in industrial applications?
  4. What regulatory, cost, and manufacturability constraints (e.g., REACH essential‑use classification, production scalabili